In [1]:
#Shortage feature engineering

In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Rishj\OneDrive\Desktop\PROJECTS\supply_chain_intelligence\data\processed\eda_cleaned_dataset.csv")
print(df.shape)

(172762, 46)


In [3]:
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

# Now this will work
df['order_week'] = df['order date (DateOrders)'].dt.to_period('W')

weekly_demand = df.groupby(['Product Name', 'order_week'])['Order Item Quantity'].sum().reset_index()
weekly_demand.columns = ['Product Name', 'order_week', 'weekly_quantity']

print(weekly_demand.shape)
print(weekly_demand.head(10))

(6972, 3)
              Product Name             order_week  weekly_quantity
0       Adult dog supplies  2017-11-27/2017-12-03              231
1       Adult dog supplies  2018-01-01/2018-01-07               78
2       Adult dog supplies  2018-01-08/2018-01-14              163
3             Baby sweater  2017-10-02/2017-10-08               80
4             Baby sweater  2017-12-11/2017-12-17               47
5             Baby sweater  2017-12-18/2017-12-24               71
6  Bag Boy Beverage Holder  2014-12-29/2015-01-04                7
7  Bag Boy Beverage Holder  2015-01-05/2015-01-11                4
8  Bag Boy Beverage Holder  2015-01-12/2015-01-18               11
9  Bag Boy Beverage Holder  2015-01-19/2015-01-25               17


In [4]:
#Worked — 6,972 product-week rows across 118 products, showing weekly quantity ordered per product.
#let's build the expanding (before-only) mean/std per product

In [5]:
# Sort by product and week so "expanding" respects time order within each product
weekly_demand = weekly_demand.sort_values(['Product Name', 'order_week']).reset_index(drop=True)

# Expanding mean/std per product, shifted so current week isn't included in its own baseline
weekly_demand['rolling_mean'] = (
    weekly_demand.groupby('Product Name')['weekly_quantity']
    .transform(lambda x: x.expanding().mean().shift(1))
)
weekly_demand['rolling_std'] = (
    weekly_demand.groupby('Product Name')['weekly_quantity']
    .transform(lambda x: x.expanding().std().shift(1))
)

print(weekly_demand.head(10))
print(weekly_demand[['rolling_mean', 'rolling_std']].isnull().sum())

              Product Name             order_week  weekly_quantity  \
0       Adult dog supplies  2017-11-27/2017-12-03              231   
1       Adult dog supplies  2018-01-01/2018-01-07               78   
2       Adult dog supplies  2018-01-08/2018-01-14              163   
3             Baby sweater  2017-10-02/2017-10-08               80   
4             Baby sweater  2017-12-11/2017-12-17               47   
5             Baby sweater  2017-12-18/2017-12-24               71   
6  Bag Boy Beverage Holder  2014-12-29/2015-01-04                7   
7  Bag Boy Beverage Holder  2015-01-05/2015-01-11                4   
8  Bag Boy Beverage Holder  2015-01-12/2015-01-18               11   
9  Bag Boy Beverage Holder  2015-01-19/2015-01-25               17   

   rolling_mean  rolling_std  
0           NaN          NaN  
1    231.000000          NaN  
2    154.500000   108.187338  
3           NaN          NaN  
4     80.000000          NaN  
5     63.500000    23.334524  
6           

In [6]:
#define the actual spike/shortage-risk threshold
# Flag a week as "shortage risk" if demand exceeds historical mean + 2 std deviations
weekly_demand['shortage_risk'] = (
    weekly_demand['weekly_quantity'] > (weekly_demand['rolling_mean'] + 2 * weekly_demand['rolling_std'])
).astype(int)

# Rows with NaN baseline (first 1-2 weeks per product) can't be judged - default to 0 (no spike, insufficient history)
weekly_demand['shortage_risk'] = weekly_demand['shortage_risk'].fillna(0).astype(int)

print(weekly_demand['shortage_risk'].value_counts())

shortage_risk
0    6620
1     352
Name: count, dtype: int64


In [7]:
#merge this shortage_risk label back onto your main order-level dataframe (df), so every order gets the label for its product+week — then you can build features and train a model, same pattern as delay prediction.
df = df.merge(
    weekly_demand[['Product Name', 'order_week', 'shortage_risk']],
    on=['Product Name', 'order_week'],
    how='left'
)

print(df['shortage_risk'].isnull().sum())  # should be 0 if merge worked cleanly
print(df.shape)

0
(172762, 48)


In [8]:
#merge in rolling_mean and rolling_std from weekly_demand
df = df.merge(
    weekly_demand[['Product Name', 'order_week', 'rolling_mean', 'rolling_std']],
    on=['Product Name', 'order_week'],
    how='left'
)

df.rename(columns={'rolling_mean': 'product_demand_mean', 'rolling_std': 'product_demand_std'}, inplace=True)

print(df[['product_demand_mean', 'product_demand_std']].isnull().sum())
print(df.shape)

product_demand_mean    4740
product_demand_std     7773
dtype: int64
(172762, 50)


In [9]:
#Fix — same pattern as before, fill with the overall dataset mean
df['product_demand_mean'] = df['product_demand_mean'].fillna(df['product_demand_mean'].mean())
df['product_demand_std'] = df['product_demand_std'].fillna(df['product_demand_std'].mean())

print(df[['product_demand_mean', 'product_demand_std']].isnull().sum())  # should be 0, 0

product_demand_mean    0
product_demand_std     0
dtype: int64


In [10]:
#let's add the order-level comparison feature — how does this specific order's quantity compare to what's typical for that product
# How many standard deviations away this order's quantity is from the product's typical order size
product_qty_stats = df.groupby('Product Name')['Order Item Quantity'].agg(['mean', 'std']).reset_index()
product_qty_stats.columns = ['Product Name', 'product_qty_mean', 'product_qty_std']

df = df.merge(product_qty_stats, on='Product Name', how='left')

# Z-score style feature: how unusual is this order's quantity for this product
df['qty_deviation'] = (df['Order Item Quantity'] - df['product_qty_mean']) / df['product_qty_std']

print(df[['Product Name', 'Order Item Quantity', 'product_qty_mean', 'product_qty_std', 'qty_deviation']].head())
print(df['qty_deviation'].isnull().sum())

   Product Name  Order Item Quantity  product_qty_mean  product_qty_std  \
0  Smart watch                     1               1.0              0.0   
1  Smart watch                     1               1.0              0.0   
2  Smart watch                     1               1.0              0.0   
3  Smart watch                     1               1.0              0.0   
4  Smart watch                     1               1.0              0.0   

   qty_deviation  
0            NaN  
1            NaN  
2            NaN  
3            NaN  
4            NaN  
75336


In [11]:
print(df['Order Item Quantity'].value_counts())
print(df['Order Item Quantity'].describe())

Order Item Quantity
1    94901
5    19510
4    19481
3    19473
2    19397
Name: count, dtype: int64
count    172762.000000
mean          2.127713
std           1.453668
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max           5.000000
Name: Order Item Quantity, dtype: float64


In [12]:
df['qty_deviation'] = df.apply(
    lambda row: 0 if row['product_qty_std'] == 0 else (row['Order Item Quantity'] - row['product_qty_mean']) / row['product_qty_std'],
    axis=1
)

print(df['qty_deviation'].isnull().sum())  # should be 0 now
print(df['qty_deviation'].describe())

0
count    1.727620e+05
mean    -6.580547e-18
std      7.507095e-01
min     -1.679341e+00
25%     -4.255773e-03
50%      0.000000e+00
75%      4.053493e-03
max      1.594478e+00
Name: qty_deviation, dtype: float64


In [13]:
category_shortage_stats = df.groupby('Category Name')['shortage_risk'].agg(['mean', 'count'])
category_shortage_stats.columns = ['shortage_rate', 'order_count']

print(category_shortage_stats.sort_values('shortage_rate', ascending=False))

                      shortage_rate  order_count
Category Name                                   
Sporting Goods             0.589286          336
Crafts                     0.526201          458
CDs                        0.505703          263
Video Games                0.366127          803
Hunting & Shooting         0.195755          424
Men's Golf Clubs           0.164234          274
Boxing & MMA               0.164179          402
Strength Training          0.155963          109
Accessories                0.155569         1697
Soccer                     0.139706          136
Lacrosse                   0.133739          329
Electronics                0.124669         3024
Tennis & Racquet           0.124204          314
Kids' Golf Clubs           0.117978          356
Golf Apparel               0.114219          429
Golf Gloves                0.113703         1029
Hockey                     0.108844          588
Girls' Apparel             0.100264         1137
Golf Bags & Carts   

In [14]:
# Sort to ensure chronological order within each product
df = df.sort_values(['Product Name', 'order date (DateOrders)']).reset_index(drop=True)

# Expanding count of prior orders for this product (before-only, avoids leakage)
df['product_order_history_count'] = (
    df.groupby('Product Name').cumcount()
)

print(df[['Product Name', 'order date (DateOrders)', 'product_order_history_count']].head(10))

         Product Name order date (DateOrders)  product_order_history_count
0  Adult dog supplies     2017-11-29 03:35:00                            0
1  Adult dog supplies     2017-11-29 03:56:00                            1
2  Adult dog supplies     2017-11-29 04:17:00                            2
3  Adult dog supplies     2017-11-29 04:38:00                            3
4  Adult dog supplies     2017-11-29 05:41:00                            4
5  Adult dog supplies     2017-11-29 06:02:00                            5
6  Adult dog supplies     2017-11-29 06:23:00                            6
7  Adult dog supplies     2017-11-29 06:44:00                            7
8  Adult dog supplies     2017-11-29 07:05:00                            8
9  Adult dog supplies     2017-11-29 07:26:00                            9


In [15]:
shortage_df = df.copy()
shortage_df.to_pickle("../data/processed/shortage_engineered_dataset.pkl")
shortage_df.to_csv("../data/processed/shortage_engineered_dataset.csv", index=False)